# CropSense AI: Soybean Hyperparameter Tuning & Model Finalization
**Project:** AI-Powered Corn and Soybean Yield Prediction System  
**Selected Champion Architecture:** Extreme Gradient Boosting (XGBoost Regressor)  
**Objective:** Fine-tune tree depth, shrinkage (learning rate), subsampling, and feature subsampling via 3-Fold Grid Search.

---
### Optimization Details
We optimize the objective `reg:squarederror` across boosting rounds and tree depths.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SOY_MODEL_PATH = r"../backend/models/soybean_model_final.pkl"
soy_artifact = joblib.load(SOY_MODEL_PATH)
print("Loaded soybean production model artifact successfully.")
print(f"Optimal Hyperparameters: {soy_artifact['best_params']}")
print("Test Set Performance Metrics:")
for k, v in soy_artifact['metrics'].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Loaded soybean production model artifact successfully.
Optimal Hyperparameters: {'colsample_bytree': 0.85, 'learning_rate': 0.07, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.85}
Test Set Performance Metrics:
  cv_r2: 0.9146
  cv_rmse: 2.9542
  cv_mae: 2.2410
  test_r2: 0.9241
  test_rmse: 2.8210
  test_mae: 2.1430
  test_mape: 5.5300


## 1. Feature Importance Rankings (Dominance of Late-Season Moisture)
Features corresponding to late August precipitation ($W133$) and topsoil available water capacity dominate tree splitting decisions.

In [2]:
model = soy_artifact['model']
feats = soy_artifact['features']
importances = model.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
plt.barh([feats[i] for i in reversed(top_idx)], [importances[i] for i in reversed(top_idx)], color='#2a9d8f')
plt.title('Top 15 Feature Importances (Tuned XGBoost Soybean Model)', fontsize=13, fontweight='bold')
plt.xlabel('Relative Feature Importance (Gain)')
plt.tight_layout()
plt.show()

Top features: W133 (August Week 33 Rainfall), W433 (August Week 33 Max Temp), W234 (Late August Vapor Pressure Deficit), Ssurface1 (Productivity), S71 (Organic Matter).


## 2. Unit Translations for Agricultural Decision-Making
Reporting the final error metrics in farmer-friendly and agronomist units:
- **Bushels per acre (bu/acre):** Traditional US commodity trade unit
- **Metric tons per hectare (t/ha):** International scientific standard ($1	ext{ bu/ac} = 0.06725	ext{ t/ha}$)
- **Quintals per acre (qtl/acre):** South Asian standard ($1	ext{ bu/ac} = 0.2722	ext{ qtl/ac}$)

In [3]:
rmse_bu = soy_artifact['metrics']['test_rmse']
mae_bu = soy_artifact['metrics']['test_mae']

print(f"Test RMSE across measurement standards:")
print(f"  US Standard:             {rmse_bu:.2f} bu/acre")
print(f"  Metric Tons per Hectare: {rmse_bu * 0.06725:.3f} t/ha")
print(f"  Quintals per Acre:       {rmse_bu * 0.2722:.2f} qtl/acre")
print(f"
Test MAE across measurement standards:")
print(f"  US Standard:             {mae_bu:.2f} bu/acre")
print(f"  Metric Tons per Hectare: {mae_bu * 0.06725:.3f} t/ha")
print(f"  Quintals per Acre:       {mae_bu * 0.2722:.2f} qtl/acre")

Test RMSE across measurement standards:
  US Standard:             2.82 bu/acre
  Metric Tons per Hectare: 0.190 t/ha
  Quintals per Acre:       0.77 qtl/acre

Test MAE across measurement standards:
  US Standard:             2.14 bu/acre
  Metric Tons per Hectare: 0.144 t/ha
  Quintals per Acre:       0.58 qtl/acre
